In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [9]:
df_raw = pd.read_csv('images_and_data/sreality_master.csv')
print(f'Dataset shape: {df_raw.shape}')
df_raw.head()

Dataset shape: (17340, 31)


,estate_id,title,locality,region,latitude,longitude,price_czk,price_per_m2,area_m2,category,...,vlastnictvi,energ_stitek,anuita,dist_vecerka_m,dist_obchod_m,dist_tram_m,dist_metro_m,dist_bus_mhd_m,url,scraped_at
0,1458614348,Prodej bytu 1+kk 36 m²,"Černá v Pošumaví, Český Krumlov",Jihočeský kraj,48.737864,14.102581,3290000.0,91389,36,1+kk,...,Osobní,C - Úsporná,NaN,723.0,NaN,NaN,NaN,481.0,https://www.sreality.cz/detail/prodej/byt/1+kk...,2026-07-18T21:41:29.040753+00:00
1,3749109836,Prodej bytu 1+kk 25 m²,"Litvínov - Horní Litvínov, Most",Ústecký kraj,50.610939,13.636989,899000.0,35960,25,1+kk,...,Družstevní,G - Mimořádně nehospodárná,NaN,28.0,1908.0,715.0,NaN,68.0,https://www.sreality.cz/detail/prodej/byt/1+kk...,2026-07-18T21:41:29.040753+00:00
2,569421900,Prodej bytu 1+kk 28 m²,"Černá v Pošumaví, Český Krumlov",Jihočeský kraj,48.737864,14.102581,2690000.0,96071,28,1+kk,...,Osobní,C - Úsporná,NaN,723.0,NaN,NaN,NaN,481.0,https://www.sreality.cz/detail/prodej/byt/1+kk...,2026-07-18T21:41:29.040753+00:00
3,4111769676,Prodej bytu 2+1 64 m²,"Ostrov, Karlovy Vary",Karlovarský kraj,50.307745,12.956295,3249000.0,50766,64,2+1,...,Osobní,C - Úsporná,NaN,231.0,1117.0,NaN,NaN,259.0,https://www.sreality.cz/detail/prodej/byt/2+1/...,2026-07-18T21:41:29.040753+00:00
4,3664875596,Prodej bytu 2+1 55 m²,"Chotěboř, Havlíčkův Brod",Kraj Vysočina,49.721356,15.671102,3390000.0,61636,55,2+1,...,Osobní,F - Velmi nehospodárná,NaN,36.0,1244.0,NaN,NaN,154.0,https://www.sreality.cz/detail/prodej/byt/2+1/...,2026-07-18T21:41:29.040753+00:00


In [10]:
# pre-split once (faster + cleaner)
stavba_parts = df_raw['stavba'].str.split(',', expand=True)
loc_parts = df_raw['locality'].str.split(',', n=1, expand=True)

df_raw = df_raw.assign(
    # --- stavba ---
    construction = stavba_parts[0].str.strip(),
    condition    = stavba_parts[1].str.strip(),
    floor        = stavba_parts[2].str.extract(r'(\d+)')[0],
    total_floors = stavba_parts[2].str.extract(r'z\s*(\d+)')[0],

    # --- locality ---
    city     = loc_parts[0].str.split(' - ').str[0],
    district = loc_parts[0].str.split(' - ').str[1].fillna(loc_parts[1])
)

# --- rules ---
# Praha → use second part (Praha 5)
df_raw.loc[df_raw['city'] == 'Praha', 'district'] = loc_parts[1]

# single-value locality → district = city
df_raw['district'] = df_raw['district'].fillna(df_raw['city'])

# --- drop columns ---
df_raw = df_raw.drop(columns=[
    'stavba','locality','estate_id','title','url',
    'scraped_at','premise','seller'
])

df_raw.head()

,region,latitude,longitude,price_czk,price_per_m2,area_m2,category,is_new,has_video,has_3d,...,dist_obchod_m,dist_tram_m,dist_metro_m,dist_bus_mhd_m,construction,condition,floor,total_floors,city,district
0,Jihočeský kraj,48.737864,14.102581,3290000.0,91389,36,1+kk,False,True,False,...,NaN,NaN,NaN,481.0,Cihlová,Novostavba,3,3,Černá v Pošumaví,Český Krumlov
1,Ústecký kraj,50.610939,13.636989,899000.0,35960,25,1+kk,False,False,False,...,1908.0,715.0,NaN,68.0,Panelová,Velmi dobrý,10,12,Litvínov,Horní Litvínov
2,Jihočeský kraj,48.737864,14.102581,2690000.0,96071,28,1+kk,False,True,False,...,NaN,NaN,NaN,481.0,Cihlová,Novostavba,3,3,Černá v Pošumaví,Český Krumlov
3,Karlovarský kraj,50.307745,12.956295,3249000.0,50766,64,2+1,False,False,False,...,1117.0,NaN,NaN,259.0,Panelová,Velmi dobrý,4,4,Ostrov,Karlovy Vary
4,Kraj Vysočina,49.721356,15.671102,3390000.0,61636,55,2+1,False,False,False,...,1244.0,NaN,NaN,154.0,Smíšená,Velmi dobrý,1,3,Chotěboř,Havlíčkův Brod


In [11]:
print('=== Missing Values ===')
missing = df_raw.isnull().sum()
print(missing[missing > 0])

print('\n=== Zero-Price Records ===')
zero_price = (df_raw['price_czk'] == 0).sum()
print(f'{zero_price} records have price = 0 ({zero_price/len(df_raw)*100:.1f}% of data)')
print('These are likely data entry errors and will be removed.')


=== Missing Values ===
prislusenstvi      1141
infrastruktura     4726
topeni             6581
telekomunikace    11236
studna            17298
vlastnictvi          11
energ_stitek       2041
anuita            17029
dist_vecerka_m       41
dist_obchod_m      2974
dist_tram_m        9605
dist_metro_m      12708
dist_bus_mhd_m       11
construction         11
condition            11
floor                11
total_floors       4652
dtype: int64

=== Zero-Price Records ===
0 records have price = 0 (0.0% of data)
These are likely data entry errors and will be removed.
